In [1]:
import bs4
import httpx

content_base_url = "https://www.gov.uk/guidance/style-guide"


def extract_style_links(html_content: bs4.BeautifulSoup):
    results = html_content.select("#manuals-frontend a.govuk-link")

    return [{"link": link.get("href")} for link in results]


async def get_style_guide():
    async with httpx.AsyncClient() as client:
        response = await client.get(f"{content_base_url}")

        soup = bs4.BeautifulSoup(response.text, "lxml")

        return extract_style_links(soup)

style_guide = await get_style_guide()

In [2]:
import asyncio
import re

from markdownify import markdownify as md

content_api_base_url = "https://www.gov.uk/api/content"

PAGES_TO_SPLIT = {
    "a-to-z": 3,
}

RULE_LENGTH_THRESHOLD = 500


async def get_section_content(page):
    async with httpx.AsyncClient() as client:
        response = await client.get(f"{content_api_base_url}{page['link']}")

        json = response.json()

        html = json["details"]["body"]

        soup = bs4.BeautifulSoup(html, "lxml")

        return {
            "title": json["title"],
            "description": json["description"],
            "content": md(str(soup), heading_style="ATX")
        }


batch_size = 10

batches = [style_guide[i : i + batch_size] for i in range(0, len(style_guide), batch_size)]

for i in range(len(batches)):
    batch = batches[i]

    contents = await asyncio.gather(*[get_section_content(page) for page in batch])

    for page, content in zip(batch, contents, strict=True):
        page["title"] = content["title"]
        page["description"] = content["description"]
        page["content"] = content["content"]

    print(f"Completed batch {i + 1} out of {len(batches)}")

    await asyncio.sleep(2)


def split_by_heading(page, level):
    slug = page["link"].split("/")[-1]
    hashes = "#" * level
    sections = []
    for part in re.split(rf"(?=^{hashes} )", page["content"], flags=re.MULTILINE):
        part = part.strip()
        if not part:
            continue
        first_line = part.split("\n")[0]
        title = first_line[level + 1:].strip() if first_line.startswith(f"{hashes} ") else "Overview"
        section_slug = re.sub(r"[^a-z0-9]+", "-", title.lower()).strip("-")
        entry_type = "rule" if len(part) >= RULE_LENGTH_THRESHOLD else "definition"
        letter = section_slug[0] if section_slug else "_"
        file = (
            f"{slug}/rules/{section_slug}.md"
            if entry_type == "rule"
            else f"{slug}/{letter}/{section_slug}.md"
        )
        sections.append({
            "link": f"{page['link']}/{section_slug}",
            "title": title,
            "description": page["description"],
            "content": part,
            "type": entry_type,
            "file": file,
        })
    return sections


for page in [p for p in style_guide if p["link"].split("/")[-1] in PAGES_TO_SPLIT]:
    level = PAGES_TO_SPLIT[page["link"].split("/")[-1]]
    sections = split_by_heading(page, level)
    style_guide.remove(page)
    style_guide.extend(sections)
    print(f"Split '{page['title']}' into {len(sections)} sections")

Completed batch 1 out of 1
Split 'A to Z' into 563 sections


In [3]:
from collections import defaultdict

def merge_definitions(pages):
    rules = [p for p in pages if p.get("type") == "rule"]
    definitions = [p for p in pages if p.get("type") == "definition"]
    other = [p for p in pages if p.get("type") not in ("rule", "definition")]

    # Group definitions by (slug, letter)
    groups = defaultdict(list)
    for defn in definitions:
        parts = defn["file"].split("/")  # e.g. ["a-to-z", "a", "abbey.md"]
        slug, letter = parts[0], parts[1]
        groups[(slug, letter)].append(defn)

    merged = []
    for (slug, letter), defs in sorted(groups.items()):
        parent_link = "/".join(defs[0]["link"].split("/")[:-1])
        combined_content = "\n\n---\n\n".join(d["content"] for d in defs)
        merged.append({
            "link": f"{parent_link}/{letter}",
            "title": f"A to Z — {letter.upper()}",
            "description": defs[0]["description"],
            "content": combined_content,
            "type": "definition",
            "file": f"{slug}/{letter}.md",
        })

    return other + rules + merged

style_guide = merge_definitions(style_guide)
print(f"After merging: {len(style_guide)} pages")


After merging: 59 pages


In [4]:
from pathlib import Path

import aiofiles

output_dir = Path("outputs/content-style-guide")
output_dir.mkdir(parents=True, exist_ok=True)


async def save_page(page):
    filepath = output_dir / page.get("file", f"{page['link'].split('/')[-1]}.md")
    filepath.parent.mkdir(parents=True, exist_ok=True)

    frontmatter = f"""---
title: {page['title']}
description: {page['description']}
---

"""

    async with aiofiles.open(filepath, "w", encoding="utf-8") as f:
        await f.write(frontmatter + page["content"])


await asyncio.gather(*[save_page(page) for page in style_guide])

print(f"\nAll {len(style_guide)} markdown files saved to {output_dir.absolute()}")


All 59 markdown files saved to /home/shaun/repos/ai/ace-pod-2-patterns/ai-uc-content-swarm/ai-uc-content-swarm/core/notebooks/outputs/content-style-guide


In [5]:
import json

index = [
    {
        "title": page["title"],
        "type": page.get("type", "rule"),
        "file": page.get("file", f"{page['link'].split('/')[-1]}.md"),
    }
    for page in style_guide
]

async with aiofiles.open(output_dir / "index.json", "w", encoding="utf-8") as f:
    await f.write(json.dumps(index))

print(f"Index saved to {(output_dir / 'index.json').absolute()}")

Index saved to /home/shaun/repos/ai/ace-pod-2-patterns/ai-uc-content-swarm/ai-uc-content-swarm/core/notebooks/outputs/content-style-guide/index.json
